# 04 — Portfolio Optimization

End-to-end validation of the DSBPSO optimizer:
1. Run a small optimization (few particles, few iterations)
2. Plot the Pareto front (CVaR vs expected ROV)
3. Inspect best-ROV and best-CVaR solutions
4. Check convergence history
5. Compare PSO-evaluation vs final re-evaluation

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.spatial.distance import cdist

from src.orchestrator import run_optimization
from src.optimizer.encoding import decode_particle
from src.data_generation.generator import generate_data
from src.data_generation.profiles import make_oil_profile, logistic_watercut
from src.valuation.capacity import compute_capped_production
from src.optimizer.evaluation import evaluate_portfolio
from src.price.schwartz_smith import generate_price_scenarios_final

# Show optimizer logs
logging.basicConfig(level=logging.INFO, format='%(name)s — %(message)s')

print('Imports OK')

## 1. Run optimization

Uses all parameters from `config/default.yaml` directly.

In [ ]:
result = run_optimization(
    config_path='../config/default.yaml',
    verbose=True,
)

archive = result['archive']
pareto_front = result['pareto_front']
config = result['config']
history = result['history']

print(f'\nArchive size: {len(archive)}')
print(f'Pareto front size: {len(pareto_front)}')

In [ ]:
# Regenerate data (same config/seed = same coordinates and costs)
field_data, host_data, cost_data = generate_data(config)
dist_matrix = cdist(field_data.coords, host_data.coords)
area = config['data_generation']['area_size_km']
n_fields = config['n_fields']
n_hosts = config['n_hosts']
n_years = config['n_years']
max_start_year = config['max_start_year']
max_wells = config['data_generation']['max_wells']
per_well_rate = config['data_generation']['per_well_rate']

# PSO-time objectives (from full archive, for convergence & PSO vs final comparison)
obj_pso = np.array([s['objectives'] for s in archive])
obj_final_archive = np.array([s['objectives_final'] for s in archive])

# Re-evaluated objectives (from pareto_front, for Pareto plots & best solutions)
obj_final = np.array([s['objectives_final'] for s in pareto_front])
best_rov_idx = np.argmax(obj_final[:, 0])
best_cvar_idx = np.argmax(obj_final[:, 1])
alloc_best_rov = decode_particle(pareto_front[best_rov_idx]['position'], n_fields, n_hosts, max_start_year, max_wells)
alloc_best_cvar = decode_particle(pareto_front[best_cvar_idx]['position'], n_fields, n_hosts, max_start_year, max_wells)

print(f'Data regenerated: {n_fields} fields, {n_hosts} hosts, {n_years} years')
print(f'Pareto front: {len(pareto_front)} solutions (from {len(archive)} archive members)')
print(f'Best ROV: idx {best_rov_idx} ({obj_final[best_rov_idx, 0]:.0f} MNOK)')
print(f'Best CVaR: idx {best_cvar_idx} ({obj_final[best_cvar_idx, 1]:.0f} MNOK)')

## 2. Pareto front

Plot CVaR vs expected ROV. Each point is a non-dominated portfolio.  
Top-right is ideal (high CVaR + high ROV = low downside risk with good return).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(obj_pso[:, 1], obj_pso[:, 0], c='steelblue', s=60, edgecolors='black', zorder=5)
ax.set_xlabel('CVaR 5% (MNOK)')
ax.set_ylabel('Expected ROV (MNOK)')
ax.set_title(f'Archive — PSO evaluation ({len(archive)} solutions, {config["price"]["n_scenarios"]} scenarios)')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.scatter(obj_final[:, 1], obj_final[:, 0], c='seagreen', s=60, edgecolors='black', zorder=5)
ax.set_xlabel('CVaR 5% (MNOK)')
ax.set_ylabel('Expected ROV (MNOK)')
ax.set_title(f'Pareto Front — Re-evaluated ({len(pareto_front)} solutions, {config["price"]["n_scenarios_final"]} scenarios)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Best solutions

Inspect the portfolios with highest ROV and highest CVaR.

In [ ]:
def describe_solution(entry, label):
    """Print a human-readable description of an archive entry."""
    alloc = decode_particle(entry['position'], n_fields, n_hosts, max_start_year, max_wells)
    obj = entry['objectives_final']
    
    print(f'\n=== {label} ===')
    print(f'Expected ROV: {obj[0]:.1f} MNOK')
    print(f'CVaR (5%):    {obj[1]:.1f} MNOK')
    print(f'Assignments:')
    for f in range(n_fields):
        host = alloc.assignment[f]
        year = alloc.start_year[f]
        wells = alloc.n_wells[f]
        if host == -1:
            print(f'  Field {f}: not developed')
        else:
            print(f'  Field {f}: Host {host}, start year {year}, {wells} wells')

describe_solution(pareto_front[best_rov_idx], 'Best Expected ROV')
describe_solution(pareto_front[best_cvar_idx], 'Best CVaR (safest)')

worst_rov_idx = np.argmin(obj_final[:, 0])
describe_solution(pareto_front[worst_rov_idx], 'Lowest ROV on front')

### Spatial view — Best ROV vs Best CVaR

Shows the tieback assignments chosen by the optimizer on the field/host map.

In [ ]:
host_colors = ['steelblue', 'tomato', 'seagreen', 'darkorange', 'purple']

def plot_tieback_map(ax, alloc, title):
    """Plot field/host positions with assigned tiebacks highlighted."""
    alloc_capex = cost_data.capex(alloc.n_wells)

    for i in range(n_fields):
        j = int(alloc.assignment[i])
        if j < 0:
            continue
        fx, fy = field_data.coords[i]
        hx, hy = host_data.coords[j]
        d = dist_matrix[i, j]
        c = alloc_capex[i, j]
        color = host_colors[j % len(host_colors)]

        ax.plot([fx, hx], [fy, hy], '-', color=color, linewidth=2.5, alpha=0.8)

        mx, my = (fx + hx) / 2, (fy + hy) / 2
        ax.annotate(f'{d:.0f} km\n{c:.0f} MNOK', (mx, my), fontsize=7,
                    ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=color, alpha=0.85))

    ax.scatter(field_data.coords[:, 0], field_data.coords[:, 1],
               s=120, c='steelblue', marker='o', zorder=5, edgecolors='white', linewidth=1.5)
    for i, name in enumerate(field_data.names):
        h = int(alloc.assignment[i])
        sy = int(alloc.start_year[i])
        w = int(alloc.n_wells[i])
        label = f'{name} (yr {sy}, {w}w)' if h >= 0 else f'{name} (skip)'
        ax.annotate(label, field_data.coords[i], fontsize=8, fontweight='bold',
                    textcoords='offset points', xytext=(8, 8), color='steelblue')

    ax.scatter(host_data.coords[:, 0], host_data.coords[:, 1],
               s=200, c='tomato', marker='s', zorder=5, edgecolors='white', linewidth=1.5)
    for j, name in enumerate(host_data.names):
        ax.annotate(name, host_data.coords[j], fontsize=8, fontweight='bold',
                    textcoords='offset points', xytext=(8, 8), color='tomato')

    ax.set_xlim(-10, area + 10)
    ax.set_ylim(-10, area + 10)
    ax.set_xlabel('x (km)')
    ax.set_ylabel('y (km)')
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

obj_rov = pareto_front[best_rov_idx]['objectives_final']
plot_tieback_map(axes[0], alloc_best_rov,
                 f'Best ROV: {obj_rov[0]:.0f} MNOK (CVaR {obj_rov[1]:.0f})')

obj_cvar = pareto_front[best_cvar_idx]['objectives_final']
plot_tieback_map(axes[1], alloc_best_cvar,
                 f'Best CVaR: {obj_cvar[1]:.0f} MNOK (ROV {obj_cvar[0]:.0f})')

plt.tight_layout()
plt.show()

## 4. Convergence history

Track archive size and best objectives over iterations.

In [ ]:
iterations = np.arange(1, len(history['archive_size']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.plot(iterations, history['archive_size'], 'o-', markersize=3)
ax.set_xlabel('Iteration')
ax.set_ylabel('Archive size')
ax.set_title('Archive Growth')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(iterations, history['best_rov'], 'o-', markersize=3, color='steelblue')
ax.set_xlabel('Iteration')
ax.set_ylabel('Best ROV (MNOK)')
ax.set_title('Best ROV Convergence')
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(iterations, history['best_cvar'], 'o-', markersize=3, color='seagreen')
ax.set_xlabel('Iteration')
ax.set_ylabel('Best CVaR (MNOK)')
ax.set_title('Best CVaR Convergence')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. PSO vs final evaluation

Do the PSO objectives (200 scenarios) agree with re-evaluation (2000 scenarios)?  
Points near the diagonal = good agreement. Scatter = noise from limited PSO scenarios.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(obj_pso[:, 0], obj_final_archive[:, 0], c='steelblue', s=40, edgecolors='black')
lims = [min(obj_pso[:, 0].min(), obj_final_archive[:, 0].min()),
        max(obj_pso[:, 0].max(), obj_final_archive[:, 0].max())]
ax.plot(lims, lims, 'k--', alpha=0.5, label='Perfect agreement')
ax.set_xlabel(f'PSO ROV ({config["price"]["n_scenarios"]} scen)')
ax.set_ylabel(f'Final ROV ({config["price"]["n_scenarios_final"]} scen)')
ax.set_title('Expected ROV: PSO vs Final')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.scatter(obj_pso[:, 1], obj_final_archive[:, 1], c='seagreen', s=40, edgecolors='black')
lims = [min(obj_pso[:, 1].min(), obj_final_archive[:, 1].min()),
        max(obj_pso[:, 1].max(), obj_final_archive[:, 1].max())]
ax.plot(lims, lims, 'k--', alpha=0.5, label='Perfect agreement')
ax.set_xlabel(f'PSO CVaR ({config["price"]["n_scenarios"]} scen)')
ax.set_ylabel(f'Final CVaR ({config["price"]["n_scenarios_final"]} scen)')
ax.set_title('CVaR: PSO vs Final')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Correlation (need >= 2 archive members)
if len(archive) >= 2:
    print(f'ROV correlation:  {np.corrcoef(obj_pso[:, 0], obj_final_archive[:, 0])[0, 1]:.3f}')
    print(f'CVaR correlation: {np.corrcoef(obj_pso[:, 1], obj_final_archive[:, 1])[0, 1]:.3f}')
else:
    print(f'Only {len(archive)} archive member(s) — correlation undefined')

## 6. Solution landscape

How many unique portfolio structures are on the front?  
Do different solutions actually differ in field assignments or just timing?

In [ ]:
print(f'Pareto front has {len(pareto_front)} solutions\n')
print(f'{"#":<4} {"ROV":>8} {"CVaR":>8}   Assignments')
print('-' * 70)

sorted_idx = np.argsort(-obj_final[:, 0])

for rank, i in enumerate(sorted_idx):
    alloc = decode_particle(pareto_front[i]['position'], n_fields, n_hosts, max_start_year, max_wells)
    obj = pareto_front[i]['objectives_final']
    
    assignments = []
    for f in range(n_fields):
        h = alloc.assignment[f]
        y = alloc.start_year[f]
        w = alloc.n_wells[f]
        if h == -1:
            assignments.append(f'F{f}:skip')
        else:
            assignments.append(f'F{f}→H{h}@y{y}({w}w)')
    
    print(f'{rank+1:<4} {obj[0]:>8.0f} {obj[1]:>8.0f}   {", ".join(assignments)}')

## 7. Well count optimization

Does the PSO optimize well counts, or just max them out?
Reference wells = data generation default (based on UR and complexity);
optimized = what the PSO chose.

In [ ]:
alloc = alloc_best_rov
ref_wells = field_data.n_wells
opt_wells = alloc.n_wells

x = np.arange(n_fields)
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, ref_wells, width, label='Reference (data gen)', color='lightgray', edgecolor='gray')
ax.bar(x + width/2, opt_wells, width, label='PSO optimized', color='steelblue', edgecolor='black')
ax.set_xlabel('Field')
ax.set_ylabel('Number of wells')
ax.set_title('Well Count: Reference vs PSO Optimized (Best ROV)')
ax.set_xticks(x)
ax.set_xticklabels(field_data.names, rotation=45, ha='right')
ax.axhline(max_wells, color='red', linestyle='--', alpha=0.5, label=f'Max wells ({max_wells})')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'Reference total wells: {ref_wells.sum()}')
print(f'Optimized total wells: {opt_wells.sum()}')
print(f'Fields at max wells: {(opt_wells >= max_wells).sum()}/{n_fields}')

## 8. Development timeline

When does each field start producing? Colored by host assignment.
Shows whether the optimizer staggers development or clusters everything at year 0.

In [ ]:
alloc = alloc_best_rov

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(n_fields):
    h = int(alloc.assignment[i])
    start = int(alloc.start_year[i])
    if h < 0:
        ax.barh(i, 1, left=start, color='lightgray', edgecolor='gray', alpha=0.5)
        ax.text(start + 0.5, i, 'skip', ha='center', va='center', fontsize=8, color='gray')
        continue

    # Build profile to find active production duration
    n_wells_f = int(alloc.n_wells[i])
    max_rate = n_wells_f * per_well_rate * field_data.complexity_factor[i]
    profile = make_oil_profile(
        n_years, float(field_data.ultimate_recovery[i]),
        max_rate, int(field_data.ramp_years[i]), float(field_data.decline_rate[i]),
    )
    # Years with meaningful production (>1% of peak)
    active_years = int((profile > 0.01 * profile.max()).sum()) if profile.max() > 0 else 1
    duration = min(active_years, n_years - start)

    color = host_colors[h % len(host_colors)]
    ax.barh(i, duration, left=start, color=color, edgecolor='black', alpha=0.8)
    ax.text(start + duration / 2, i, f'H{h}, {n_wells_f}w',
            ha='center', va='center', fontsize=8, fontweight='bold', color='white')

ax.set_yticks(range(n_fields))
ax.set_yticklabels(field_data.names)
ax.set_xlabel('Year')
ax.set_title('Development Timeline (Best ROV)')
ax.set_xlim(-0.5, n_years + 0.5)
ax.grid(True, alpha=0.3, axis='x')

# Legend for hosts
used_hosts = sorted(set(int(h) for h in alloc.assignment if h >= 0))
legend_elements = [Patch(facecolor=host_colors[h], edgecolor='black', label=host_data.names[h])
                   for h in used_hosts]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

## 9. Production profiles

Oil production per field (after capping) for the best ROV solution.
Built from PSO-optimized n_wells, shifted by start_year, capped by host capacity.

In [ ]:
alloc = alloc_best_rov

# Build oil profiles from allocation.n_wells (same logic as evaluation.py)
oil_profiles = np.zeros((n_fields, n_years))
max_rates = np.zeros(n_fields)

for f in range(n_fields):
    if alloc.assignment[f] < 0:
        continue
    n_wells_f = int(alloc.n_wells[f])
    max_rate = n_wells_f * per_well_rate * field_data.complexity_factor[f]
    max_rates[f] = max_rate

    oil_profiles[f] = make_oil_profile(
        n_years, float(field_data.ultimate_recovery[f]),
        max_rate, int(field_data.ramp_years[f]), float(field_data.decline_rate[f]),
    )

# Precompute water cut fractions at field-relative time
t = np.arange(n_years, dtype=float)
wc_fractions = np.zeros((n_fields, n_years))
for f in range(n_fields):
    if alloc.assignment[f] < 0:
        continue
    wc_fractions[f] = logistic_watercut(
        t, float(field_data.wc_midpoint[f]),
        float(field_data.wc_initial[f]),
        float(field_data.wc_final[f]),
        float(field_data.wc_steepness[f]),
    )

# Compute capped production (FCFS oil + derived gas/water)
capped = compute_capped_production(
    oil_profiles, host_data, alloc, n_years,
    max_rates=max_rates,
    ramp_years=field_data.ramp_years,
    gor=field_data.gor,
    wc_fractions=wc_fractions,
)

# Reassemble capped oil into (n_fields, n_years)
capped_oil_all = np.zeros((n_fields, n_years))
for host_idx, host_result in capped.items():
    for local_idx, field_idx in enumerate(host_result['field_indices']):
        capped_oil_all[field_idx] = host_result['oil'][local_idx]

# Stacked area chart
years = np.arange(n_years)
fig, ax = plt.subplots(figsize=(12, 6))

# Sort fields by start year for cleaner stacking
developed = [f for f in range(n_fields) if alloc.assignment[f] >= 0]
developed_sorted = sorted(developed, key=lambda f: alloc.start_year[f])

colors = plt.cm.tab10(np.linspace(0, 1, n_fields))
bottom = np.zeros(n_years)
for f in developed_sorted:
    ax.fill_between(years, bottom, bottom + capped_oil_all[f],
                     alpha=0.7, color=colors[f], label=field_data.names[f])
    bottom += capped_oil_all[f]

ax.set_xlabel('Year')
ax.set_ylabel('Oil production (MSm3/y)')
ax.set_title('Capped Oil Production by Field (Best ROV)')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

total_oil = capped_oil_all.sum()
print(f'Total portfolio oil production: {total_oil:.1f} MSm3 ({total_oil * 6.29:.0f} mmbbl)')

## 10. Host capacity utilization

Oil demand vs capacity per host. Shaded red area = production lost to capping.
If demand never hits capacity, capping is not a binding constraint.

In [ ]:
alloc = alloc_best_rov
years = np.arange(n_years)

# Build uncapped shifted profiles per host (original demand without deferral)
uncapped_per_host = {}
for host_idx in range(n_hosts):
    fields_on_host = [f for f in range(n_fields)
                      if alloc.assignment[f] == host_idx]
    if not fields_on_host:
        continue

    uncapped_oil = np.zeros(n_years)
    for f in fields_on_host:
        start = int(alloc.start_year[f])
        profile = oil_profiles[f]
        usable = min(len(profile), n_years - start)
        if usable > 0:
            uncapped_oil[start:start + usable] += profile[:usable]

    uncapped_per_host[host_idx] = uncapped_oil

active_hosts = sorted(uncapped_per_host.keys())
n_active = len(active_hosts)

fig, axes = plt.subplots(1, n_active, figsize=(5 * n_active, 5), squeeze=False)

for plot_idx, host_idx in enumerate(active_hosts):
    ax = axes[0, plot_idx]
    capacity = host_data.oil_capacity[host_idx]
    demand = uncapped_per_host[host_idx]

    # Per-field capped production for this host
    fields_here = []
    field_capped = []
    if host_idx in capped:
        for li, fi in enumerate(capped[host_idx]['field_indices']):
            fields_here.append(fi)
            field_capped.append(capped[host_idx]['oil'][li])

    # Stacked area per field (shows which field produces when)
    colors = plt.cm.tab10(np.linspace(0, 1, n_fields))
    bottom = np.zeros(n_years)
    for fi, fc in zip(fields_here, field_capped):
        ax.fill_between(years, bottom, bottom + fc,
                        alpha=0.7, color=colors[fi], label=field_data.names[fi])
        bottom += fc

    # Original demand (no deferral) and capacity
    ax.plot(years, demand, 'k--', linewidth=1.5, label='Original demand (no deferral)')
    ax.plot(years, capacity, 'r-', linewidth=2, label='Capacity')

    ax.set_title(f'{host_data.names[host_idx]} ({len(fields_here)} fields)')
    ax.set_xlabel('Year')
    ax.set_ylabel('Oil (MSm3/y)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Host Capacity Utilization (Best ROV)', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. CAPEX breakdown

Per-field CAPEX split into components: drilling (wells × depth-based cost),
subsea infrastructure, and pipeline (distance-based). Only developed fields shown.

In [ ]:
alloc = alloc_best_rov
developed = [f for f in range(n_fields) if alloc.assignment[f] >= 0]

drilling_costs = []
subsea_costs = []
pipeline_costs = []
labels = []

for f in developed:
    h = int(alloc.assignment[f])
    n_w = int(alloc.n_wells[f])
    drill = float(cost_data.well_cost_per_well[f]) * n_w
    subsea = cost_data.subsea_base
    pipe = float(cost_data.pipeline_cost[f, h])

    drilling_costs.append(drill)
    subsea_costs.append(subsea)
    pipeline_costs.append(pipe)
    labels.append(f'{field_data.names[f]}\n({n_w}w, {field_data.depth[f]:.0f}m)')

x = np.arange(len(developed))
fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(x, drilling_costs, label='Drilling', color='steelblue', edgecolor='black')
ax.bar(x, subsea_costs, bottom=drilling_costs, label='Subsea', color='seagreen', edgecolor='black')
bottom2 = np.array(drilling_costs) + np.array(subsea_costs)
ax.bar(x, pipeline_costs, bottom=bottom2, label='Pipeline', color='sandybrown', edgecolor='black')

# Total label on top
totals = np.array(drilling_costs) + np.array(subsea_costs) + np.array(pipeline_costs)
for i, total in enumerate(totals):
    ax.text(i, total + 50, f'{total:.0f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('CAPEX (MNOK)')
ax.set_title('CAPEX Breakdown by Field (Best ROV)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'Total portfolio CAPEX: {totals.sum():.0f} MNOK')
print(f'  Drilling: {sum(drilling_costs):.0f} MNOK ({sum(drilling_costs)/totals.sum()*100:.0f}%)')
print(f'  Subsea:   {sum(subsea_costs):.0f} MNOK ({sum(subsea_costs)/totals.sum()*100:.0f}%)')
print(f'  Pipeline: {sum(pipeline_costs):.0f} MNOK ({sum(pipeline_costs)/totals.sum()*100:.0f}%)')

## 12. Field-level ROV

Per-field expected ROV contribution for the best ROV solution.
Negative values mean the field destroys value (CAPEX > revenue).

In [ ]:
# Re-evaluate best ROV solution to get per-field values
prices_eval = generate_price_scenarios_final(config)
result_rov = evaluate_portfolio(alloc_best_rov, field_data, host_data, cost_data, prices_eval, config)

field_rovs = result_rov.field_values

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['steelblue' if v >= 0 else 'tomato' for v in field_rovs]
y = np.arange(n_fields)
ax.barh(y, field_rovs, color=colors, edgecolor='black')
ax.set_yticks(y)
ax.set_yticklabels([f'{field_data.names[f]} (H{alloc_best_rov.assignment[f]}, {alloc_best_rov.n_wells[f]}w)'
                     if alloc_best_rov.assignment[f] >= 0
                     else f'{field_data.names[f]} (skip)'
                     for f in range(n_fields)], fontsize=9)
ax.set_xlabel('Expected ROV (MNOK)')
ax.set_title('Per-Field ROV Contribution (Best ROV Solution)')
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(True, alpha=0.3, axis='x')

# Annotate values
for i, v in enumerate(field_rovs):
    offset = max(abs(field_rovs).max() * 0.02, 50)
    ax.text(v + (offset if v >= 0 else -offset), i, f'{v:.0f}', va='center',
            ha='left' if v >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.show()

print(f'Portfolio E[ROV]: {result_rov.expected_rov:.0f} MNOK')
print(f'Sum of field ROVs: {field_rovs.sum():.0f} MNOK')